# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [3]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [4]:
# Initialization

load_dotenv(r"C:\Users\Dell\Desktop\.env", override = True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [5]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [6]:
# This function looks rather simpler than the one from my video, because we're taking advantage of the latest Gradio updates

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872

To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [7]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
ticket_discounts={"london":5, "tokyo":15}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")
def get_ticket_discount(destination_city):
    print(f"Tool get_ticket_discount called for {destination_city}")
    city = destination_city.lower()
    return ticket_discounts.get(city,0)

In [8]:
get_ticket_price("Berlin")
get_ticket_discount("Berlin")

Tool get_ticket_price called for Berlin
Tool get_ticket_discount called for Berlin


0

In [9]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

discount_function = {
    "name": "get_ticket_discount",
    "description": "Get the discount on price of a return ticket to the destination city. Call this whenever you need to know the discount on the ticket price, for example when a customer asks 'Is there a discount on the price on the ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The discount on price to the city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [10]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function},
         {"type":"function", "function": discount_function}]
tools_functions_map = {
    "get_ticket_price":get_ticket_price,
    "get_ticket_discount":get_ticket_discount
}

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f"Response is{response}")
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_responses, city = handle_tool_call(message)
        messages.append(message)
        for tool_response in tool_responses:
            messages.append(tool_response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [16]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_calls = message.tool_calls;
    arguments = json.loads(tool_calls[0].function.arguments)
    city = arguments.get('destination_city')
    responses=[]
    
    for tool_call in tool_calls:
        name = tool_call.function.name
        if name in tools_functions_map:
            key = "price" if "price" in name else "discount"
            value = tools_functions_map[name](city)
            responses.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city, key : value}),
                "tool_call_id": tool_call.id
            })
    return responses, city

In [17]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874

To create a public link, set `share=True` in `launch()`.


Response isChatCompletion(id='chatcmpl-AlXRb0uz4RP5c7XJdUQtOY09bHcxa', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_bAxmTTYUpUlLLHdt6OHPxVpb', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')]))], created=1735893339, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier=None, system_fingerprint='fp_0aa8d3e20b', usage=CompletionUsage(completion_tokens=17, prompt_tokens=212, total_tokens=229, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Tool get_ticket_price called for London
